# **1. Cargue de KML, segmentación y creación de los bounding box**

Este script permite cargar un archivo KML con rutas (tipo `LineString`), segmentarlas en tramos de longitud fija (por ejemplo, 100 metros) y calcular un *bounding box* (caja envolvente) alrededor de cada segmento. Las principales funcionalidades del proceso son:

- 📥 **Lectura del archivo KML**: Se extraen los elementos `Placemark` con geometría y nombre desde un archivo `.kml`.
- 📏 **Cálculo de longitud geodésica**: Se utiliza `pyproj.Geod` para calcular la longitud precisa de cada ruta.
- ✂️ **Segmentación de rutas**: Cada línea se divide en múltiples segmentos de longitud aproximada definida por el usuario (e.g., 100 m).
- 📦 **Creación de bounding boxes**: Se genera un *bounding box* alrededor de cada segmento extendido unos metros adicionales para análisis espacial.
- 🗂️ **Exportación opcional**: Los resultados se pueden guardar en CSV, GeoJSON o visualizarse en un mapa interactivo (`leafmap`).

In [ ]:
import os,glob
from tqdm import tqdm
import geopandas as gpd   
import pandas as pd
from shapely.geometry import Point, LineString, Polygon
from xml.etree import ElementTree as ET
import math
from shapely.ops import transform
import pyproj
import leafmap

# ---------------------------------------------
# PARÁMETROS INICIALES
# ---------------------------------------------
metros=100
extension_in_meters = 18#15  # Define la distancia en metros para el bounding Box

root_folder= './data' # Reemplaza con la ruta de tu archivo
ruta_kml = os.path.join(root_folder,'Troncales_V.kml') # Reemplazar con el nombre del archivo kml
meters_to_degrees = extension_in_meters / 111320  # Conversión aproximada de metros a grados (latitud)

## FUNCIONES
def calcular_bounding_box(linestring, extension):
    if linestring.is_empty:
        return None

    # Obtener los límites del LineString
    minx, miny, maxx, maxy = linestring.bounds

    # Extender los límites
    minx -= extension
    miny -= extension
    maxx += extension
    maxy += extension

    # Crear el bounding box como lista
    return [minx, miny, maxx, maxy]

def calcular_longitud(line):
    coords = list(line.coords)
    total_length_m = 0.0
    for i in range(len(coords)-1):
        lon1, lat1 = coords[i][0], coords[i][1]
        lon2, lat2 = coords[i+1][0], coords[i+1][1]
        _, _, dist = geod.inv(lon1, lat1, lon2, lat2)
        total_length_m += dist
    return total_length_m

# Función para eliminar la Z de las geometrías (opcional si existen Z)
def drop_z(geom):
    if geom.has_z:
        return LineString([(x, y) for (x, y, z) in geom.coords])
    else:
        return geom

# ---------------------------------------------
# PARSEO DEL ARCHIVO KML
# ---------------------------------------------
tree = ET.parse(ruta_kml)
root = tree.getroot()

# Definir el namespace del KML
namespaces = {'kml': 'http://www.opengis.net/kml/2.2'}

# Extraer todos los placemarks
placemarks = root.findall(".//kml:Placemark", namespaces)

# Inicializar listas para almacenar datos
names = []
geometries = []

# Procesar cada placemark
for placemark in placemarks:
    # Obtener el nombre
    name = placemark.find("kml:name", namespaces)
    name = name.text if name is not None else "Sin nombre"

    # Obtener las coordenadas
    coordinates = placemark.find(".//kml:coordinates", namespaces)
    if coordinates is not None:
        coord_text = coordinates.text.strip()
        coords = [
            tuple(map(float, coord.split(',')))
            for coord in coord_text.split()
        ]

        # Determinar el tipo de geometría según la cantidad de puntos
        if len(coords) == 1:
            geometries.append(Point(coords[0]))
        else:
            geometries.append(LineString(coords))
    else:
        geometries.append(None)

    # Agregar el nombre
    names.append(name)

# Crear un GeoDataFrame
gdf = gpd.GeoDataFrame({'Name': names, 'geometry': geometries}, crs="EPSG:4326")
#gdf.to_file("salida.geojson", driver="GeoJSON")

# Filtrar por geometrías de tipo Polygon
gdf_rutas = gdf[gdf.geometry.apply(lambda geom: isinstance(geom, LineString))]

# Opcional: Guardar el GeoDataFrame filtrado
out_geojson=ruta_kml.replace('.kml','_mt{}_ext{}.geojson'.format(metros,extension_in_meters))
out_shp=out_geojson.replace('.geojson','.shp')
#gdf_rutas.to_file(out_geojson, driver="GeoJSON")
#gdf_rutas.to_file(out_shp)

gdf_rutas = gdf_rutas.to_crs(epsg=4326)


# ---------------------------------------------
# CÁLCULO DE LONGITUDES Y SEGMENTACIÓN
# ---------------------------------------------
geod = pyproj.Geod(ellps='WGS84')

# Aplica la función a cada geometría del gdf
gdf_rutas['length'] = gdf_rutas.geometry.apply(calcular_longitud)


# Proyectar a EPSG:3857 para medir en metros
gdf_rutas_3857 = gdf_rutas.copy()
gdf_rutas_3857['geometry'] = gdf_rutas_3857['geometry'].apply(drop_z)
gdf_rutas_3857 = gdf_rutas_3857.to_crs(epsg=3857)

segmentos = []

# Recorremos cada fila (cada línea)
for idx, row in gdf_rutas_3857.iterrows():
    line = row.geometry
    length_m = line.length
    name = row['Name']  # Conservar la columna Name
    # Numero de segmentos = entero superior de la longitud/1000
    n_segments = math.ceil(length_m / metros)

    # Distancias a lo largo de la línea donde obtendremos puntos (cada 1000 m)
    distances = [i*metros for i in range(n_segments)]
    if distances[-1] < length_m:
        distances.append(length_m)

    # Interpolamos puntos a cada distancia
    points = [line.interpolate(d) for d in distances]

    # Creamos segmentos de ~1 km
    for i in range(len(points)-1):
        seg = LineString([points[i], points[i+1]])
        seg_length_km = seg.length  # Convertir la longitud del segmento a km
        # Conservar original_idx, Name y asignar length_km
        segmentos.append((idx, seg, seg_length_km, name))

# Creamos un nuevo gdf con los segmentos, su longitud y el nombre original
gdf_segmentos = gpd.GeoDataFrame(segmentos, columns=['original_idx', 'geometry', 'length', 'Name'], crs=gdf_rutas_3857.crs)

# Reproyectamos de vuelta a EPSG:4326
gdf_segmentos = gdf_segmentos.to_crs(epsg=4326)

# Crear la columna 'bb' en el GeoDataFrame
gdf_segmentos['bb'] = gdf_segmentos.geometry.apply(lambda geom: calcular_bounding_box(geom, meters_to_degrees))

# Ahora gdf_segmentos tiene aproximadamente 1km por segmento, una columna length_km con su distancia, y conserva la columna Name
print(gdf_segmentos.shape,gdf_segmentos.columns)
print(gdf_segmentos['original_idx'].value_counts())
print(gdf_segmentos['Name'].value_counts())
display(gdf_segmentos.head())

# Exportar un CSV: convertir geometría a WKT SIN modificar el GeoDataFrame original
out_csv = out_geojson.replace('.geojson', '.csv')

# Creamos una copia para exportar a CSV
gdf_segmentos_csv = gdf_segmentos.copy()
gdf_segmentos_csv['geometry'] = gdf_segmentos_csv['geometry'].apply(lambda x: x.wkt)

# Exportamos la copia con geometría como texto
#gdf_segmentos_csv.to_csv(out_csv, index=False)

# ---------------------------------------------
# VISUALIZACIÓN EN MAPA INTERACTIVO
# ---------------------------------------------
# Visualizar en mapa interactivo: usamos el GeoDataFrame original con geometría válida
m = leafmap.Map()
m.add_gdf(gdf_segmentos, layer_name="Capa KML")  # Este sí tiene geometría activa y CRS
m


# **2. Descarga automatizada de imágenes satelitales por tramos**

Este script automatiza la descarga de imágenes georreferenciadas desde servicios como *Terrain* o *Satellite* utilizando `leafmap`, tomando como base los segmentos generados previamente (`gdf_segmentos`). Cada imagen corresponde al *bounding box* de un tramo específico de una troncal seleccionada.

### ✅ Funcionalidades clave:

- 🔍 **Filtrado por troncales**: Permite seleccionar qué troncales (según `original_idx`) se desean procesar.
- 🗺️ **Descarga según fuente y zoom**: Soporta múltiples fuentes de imágenes (`Terrain`, `Satellite`, etc.) y niveles de zoom personalizables.
- 🧠 **Evita descargas duplicadas**: Antes de descargar, verifica si la imagen ya existe en disco para evitar sobreescritura innecesaria.
- 🛰️ **Uso de `leafmap.tms_to_geotiff`**: Utiliza esta función para descargar las imágenes directamente desde servicios de mapas y guardarlas como archivos `.tif`.

### 🛠️ Parámetros configurables:
- `troncales`: lista de índices de troncales a procesar.
- `zooms`: niveles de zoom deseados para las imágenes.
- `sources`: fuentes de imágenes a utilizar (e.g., `Terrain`, `Satellite`).
- `metros`, `extension_in_meters`: definen el tamaño de los segmentos y la extensión del área a capturar por imagen.

In [ ]:
#DESCARGAR TODAS LAS IMÁGENES CHEQUEANDO SI YA FUE DESCARGADA
import os
from tqdm import tqdm
import leafmap

# Definir parámetros
zooms = [21]
troncales = [16, 0, 19, 3,11, 7]#ids de las troncales
troncales = [0]
sources = ['Terrain'#'Satellite',
           ]
images_fol = './data'#Reemplazar 


# Loop para descargar imágenes
for z in zooms:
    print(z)
    for source in sources:
        for troncal in troncales:
            gdf_segmentos_fil = gdf_segmentos[gdf_segmentos['original_idx'] == troncal]
            print(gdf_segmentos.shape)
            print(f"Procesando troncal: {troncal}")

            for idx, row in tqdm(gdf_segmentos_fil.iterrows(), total=gdf_segmentos_fil.shape[0], desc=f"Zoom: {z}, Source: {source}, Troncal: {troncal}"):

                # Crear el nombre de la carpeta principal basado en los parámetros
                folder_name = f"Source{source}_Z{z}_SegmentMts{metros}_ExtBB{extension_in_meters}"
                folder_name = os.path.join(images_fol, folder_name)

                # Agregar una subcarpeta basada en la columna 'Name'
                name_folder = os.path.join(folder_name, str(row['Name'].split('_')[0]))

                # Obtener el bounding box y generar un nombre único para la imagen
                bbox = row['bb']  # Bounding box de la columna 'bb'
                image_name = os.path.join(name_folder, f"{row['Name'].split('_')[0]}_{idx}.tif")  # Nombre único para cada imagen
                print(image_name)  # Imprime la ruta de la imagen para depuración
                os.makedirs(name_folder, exist_ok=True)  # Asegura que la carpeta existe

                # Verificar si la imagen ya existe
                if not os.path.exists(image_name):
                    try:
                        # Descargar la imagen usando leafmap
                        leafmap.tms_to_geotiff(image_name, bbox, zoom=z, source=source)
                        print(f"Imagen guardada: {image_name}")
                    except Exception as e:
                        print(f"Error al descargar la imagen {image_name}: {e}")
                else:
                    print(f"Imagen ya existe: {image_name}. Omitiendo descarga.")
